# Baseline TabICLv2 — búsqueda de HP solo ICL

Mismo baseline y split de la referencia Retouche; entrenamiento de la prueba de capacidad ICL.
Se actualiza **solo ICL**, sin adaptador. Columnas y filas permanecen congeladas.

- El YAML define el grid, las épocas, los cortes y la ejecución: `tabicl.icl` y `tabicl.ejecucion`.
- Split estratificado 70/30. Después se aplica el mismo undersampling a ambos conjuntos: todas las bajas y 10% de CONTINUA. Las filas y el contexto quedan fijos durante toda la búsqueda.
- Una sola semilla. Cada combinación comienza con los pesos preentrenados y estados nuevos de AdamW, scaler y scheduler. El shuffle y los splits contexto/consulta de los batches se repiten entre combinaciones.
- Validación por **AUC**. Época 0 una sola vez; luego los cortes del YAML. **202106 no se utiliza para entrenar, validar ni seleccionar HP.**

Ejecutar en un kernel CUDA nuevo, con el `config_tfm.yaml` actualizado junto a la notebook.
No hay etapa de producción, guardado de pesos ni archivos de predicciones en esta notebook.

## 1. Configuración

In [1]:
import os
import gc
import json
import math
import random
import tempfile
import time
from contextlib import ExitStack
from copy import deepcopy
from datetime import datetime
from itertools import product
from pathlib import Path
from collections import OrderedDict

import yaml
import numpy as np
import pandas as pd

import traceback


In [2]:
ARCHIVO_CONFIG = Path("config_tfm.yaml").resolve()
with ARCHIVO_CONFIG.open(encoding="utf-8") as archivo:
    config = yaml.safe_load(archivo)

PARAM = {
    "training": deepcopy(config["training"]),
    "fijos": deepcopy(config["tabicl"]["icl"]["fijos"]),
    "ejecucion": deepcopy(config["tabicl"]["ejecucion"]),
}
hs = config["tabicl"]["icl"]["hs"]
SEED = int(PARAM["training"]["semilla_primigenia"][0])

DATA_DIR = (ARCHIVO_CONFIG.parent / config["directorios"]["datasets"]).resolve()
EXP_DIR = (ARCHIVO_CONFIG.parent / config["directorios"]["exp"] / config["exp_name"]).resolve()
assert DATA_DIR.is_dir(), DATA_DIR
EXP_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVO_LOG = EXP_DIR / "Grid_log.txt"
print("Experimento:", EXP_DIR)

assert set(hs) == {"lr_maximo", "weight_decay"}
assert PARAM["fijos"]["metrica_seleccion"] == "auc"
assert PARAM["fijos"]["freeze_col"] and PARAM["fijos"]["freeze_row"]
assert PARAM["fijos"]["n_estimators_finetune"] == PARAM["fijos"]["n_estimators_inference"] == 1
assert PARAM["fijos"]["preprocesamiento"] == {
    "norm_methods": "none", "feat_shuffle_method": "none", "class_shuffle_method": "none"
}
assert PARAM["fijos"]["optimizacion"]["scheduler"] == "coseno"
assert 0 <= PARAM["fijos"]["optimizacion"]["warmup_epochs"] < PARAM["fijos"]["epochs"]
assert all(0 < epoca <= PARAM["fijos"]["epochs"] for epoca in PARAM["fijos"]["evaluar_epochs"])


Experimento: /home/marco/code/DMEyF/exp/TabICL_Solo_ICL


In [3]:
# Configurar antes de importar PyTorch e inicializar CUDA.
ejecucion = PARAM["ejecucion"]
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ejecucion["cublas_workspace_config"]
for nombre in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[nombre] = str(ejecucion["hilos_cpu"])

import torch
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tabicl import FinetunedTabICLClassifier
from tabicl._finetune.data import _build_meta_batch, move_meta_batch
from tabicl._sklearn.preprocessing import TransformToNumerical, EnsembleGenerator

DEVICE = torch.device(ejecucion["dispositivo"])
assert torch.cuda.is_available(), "Seleccionar el entorno PyTorch CUDA."
assert ejecucion["precision_evaluacion"] == "fp32"
torch.cuda.set_device(DEVICE)
torch.set_num_threads(ejecucion["hilos_cpu"])
torch.backends.cuda.matmul.allow_tf32 = ejecucion["permitir_tf32"]
torch.backends.cudnn.allow_tf32 = ejecucion["permitir_tf32"]
torch.backends.cudnn.deterministic = ejecucion["cudnn_deterministic"]
torch.use_deterministic_algorithms(
    ejecucion["algoritmos_deterministas"], warn_only=ejecucion["determinismo_warn_only"]
)

def fijar_semilla(semilla):
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)
    torch.cuda.manual_seed_all(semilla)

fijar_semilla(SEED)
print(torch.cuda.get_device_name(DEVICE))


def limitar_vram():
    # Mismo margen del ensayo de capacidad; no modifica el sistema.
    torch.cuda.empty_cache()
    libre, total = torch.cuda.mem_get_info(DEVICE)
    capacidad = libre + torch.cuda.memory_reserved(DEVICE)
    limite = capacidad - ejecucion["margen_vram_gib"]*1024**3
    assert limite > torch.cuda.memory_allocated(DEVICE), "VRAM disponible insuficiente."
    torch.cuda.set_per_process_memory_fraction(min(1.0, limite/total), DEVICE)


[transformers] Disabling PyTorch because PyTorch >= 2.6 is required but found 2.5.1
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


NVIDIA GeForce RTX 3090


## 2. Dataset baseline — misma construcción de la referencia

In [4]:
historia = pd.read_csv(DATA_DIR / PARAM["training"]["dataset"], low_memory=False)
historia = historia.loc[historia["foto_mes"] <= max(PARAM["training"]["future"])]
historia = historia.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
campos_lag = [c for c in historia.select_dtypes(include="number").columns
              if c not in PARAM["training"]["campos_excluidos"]]

# Variable numérica toda cero y sin NA en un mes: convertir esos valores a NA.
for campo in campos_lag:
    todo_cero = historia[campo].eq(0).groupby(historia["foto_mes"]).all()
    historia.loc[historia["foto_mes"].isin(todo_cero.index[todo_cero]), campo] = np.nan

print("Historial:", historia.shape)


Historial: (4378500, 155)


In [5]:
# Materializar una sola matriz CPU; conservar el baseline completo en RAM.
campos_base = [c for c in historia.columns
               if c not in {"clase_ternaria", "clase01", "fold", "azar", "training"}]
columnas = (campos_base + [f"{c}_lag1" for c in campos_lag]
            + [f"{c}_lag2" for c in campos_lag]
            + [f"{c}_delta{k}" for c in campos_lag for k in (1, 2)])
X_baseline = np.empty((len(historia), len(columnas)), dtype=np.float64, order="F")
posiciones = {c: i for i, c in enumerate(columnas)}
for campo in campos_base:
    X_baseline[:, posiciones[campo]] = historia[campo].to_numpy(dtype=np.float64)
for campo in campos_lag:
    actual = historia[campo].to_numpy(dtype=np.float64)
    grupo = historia.groupby("numero_de_cliente", sort=False)[campo]
    for k in (1, 2):
        lag = grupo.shift(k).to_numpy(dtype=np.float64)
        X_baseline[:, posiciones[f"{campo}_lag{k}"]] = lag
        X_baseline[:, posiciones[f"{campo}_delta{k}"]] = actual - lag

metadatos = historia[["numero_de_cliente", "foto_mes", "clase_ternaria"]].copy()
metadatos["clase_ternaria"] = metadatos["clase_ternaria"].astype("category")
metadatos["clase01"] = metadatos["clase_ternaria"].isin(["BAJA+1", "BAJA+2"]).astype(np.int64)
dataset_baseline = pd.DataFrame(X_baseline, columns=columnas, copy=False)
del historia, grupo, lag, actual
gc.collect()
print("Baseline completo en RAM:", dataset_baseline.shape)
print("Columnas para entrenar:", X_baseline.shape[1], "(referencia: 762)")


Baseline completo en RAM: (4378500, 762)
Columnas para entrenar: 762 (referencia: 762)


## 3. Split y muestras fijas

La partición se realiza sobre `training.train`, antes del undersampling, igual que en la referencia.
La muestra de CONTINUA se elige una sola vez para cada partición; no se vuelve a sortear por combinación ni por época.
La clase positiva sigue agrupando BAJA+1 y BAJA+2.

In [6]:
def seleccionar_training(indices, semilla, training):
    # Mismo criterio de undersampling que LGBM/XGB: todas las bajas y 10% de CONTINUA.
    azar = np.random.default_rng(semilla).random(len(indices))
    bajas = metadatos.loc[indices, "clase_ternaria"].isin(["BAJA+1", "BAJA+2"]).to_numpy()
    return indices[(azar <= training["undersampling"]) | bajas]


In [7]:
indices = metadatos.index[metadatos["foto_mes"].isin(PARAM["training"]["train"])].to_numpy()
idx_train, idx_valid = train_test_split(
    indices, train_size=PARAM["training"]["training_pct"] / 100,
    stratify=metadatos.loc[indices, "clase_ternaria"], random_state=SEED,
)
idx_train = seleccionar_training(idx_train, SEED, PARAM["training"])
# Una muestra fija del 30%: mismo factor de undersampling que en training.
idx_valid = seleccionar_training(idx_valid, SEED, PARAM["training"])
assert not np.intersect1d(idx_train, idx_valid).size
assert not set(PARAM["training"]["train"]) & set(PARAM["training"]["future"])
print("Train con undersampling:", len(idx_train))
print("Validación con undersampling:", len(idx_valid))


Train con undersampling: 306255
Validación con undersampling: 131152


## 4. Preprocesamiento de la API

Para entrenar se imputa con `TransformToNumerical`, ajustado solo con training. Cada bloque pasa por `_build_meta_batch`, como en la prueba ICL de capacidad: la API construye sus contextos/consultas y ajusta su preprocesamiento.

Para validar se prepara una vez el contexto completo de training con `TransformToNumerical` y `EnsembleGenerator`; validación reutiliza ese ajuste. Estas matrices CPU se conservan para todas las combinaciones. No hay transformaciones adicionales propias.

In [8]:
def preparar_datos(indices_train, indices_eval, semilla, parametros):
    numerico = TransformToNumerical()
    X_numerico = numerico.fit_transform(X_baseline[indices_train])
    y = metadatos.loc[indices_train, "clase01"].to_numpy()
    generador = EnsembleGenerator(
        classification=True,
        n_estimators=parametros["fijos"]["n_estimators_finetune"],
        random_state=semilla, **parametros["fijos"]["preprocesamiento"],
    )
    generador.fit(X_numerico, y)
    metodo = parametros["fijos"]["preprocesamiento"]["norm_methods"]
    X_train = np.ascontiguousarray(generador.preprocessors_[metodo].X_transformed_, dtype=np.float32)
    generador.X_ = generador.y_ = None
    generador.preprocessors_[metodo].X_transformed_ = None
    del X_numerico
    gc.collect()

    X_eval = np.empty((len(indices_eval), X_train.shape[1]), dtype=np.float32)
    bloque = parametros["ejecucion"]["tamanos_consulta_oom"][0]
    for desde in range(0, len(indices_eval), bloque):
        hasta = min(desde + bloque, len(indices_eval))
        entrada = numerico.transform(X_baseline[indices_eval[desde:hasta]])
        X_eval[desde:hasta] = generador.transform(entrada, mode="test")[metodo][0][0]
    assert np.isfinite(X_train).all() and np.isfinite(X_eval).all()
    return X_train, y, X_eval


In [9]:
numerico_train = TransformToNumerical()
X_train = np.ascontiguousarray(
    numerico_train.fit_transform(X_baseline[idx_train]), dtype=np.float64
)
y_train = metadatos.loc[idx_train, "clase01"].to_numpy(dtype=np.int64)
del numerico_train
X_contexto, y_contexto, X_valid = preparar_datos(idx_train, idx_valid, SEED, PARAM)
y_valid = metadatos.loc[idx_valid, "clase01"].to_numpy(dtype=np.int64)
assert np.array_equal(y_train, y_contexto)
gc.collect()
print("Training:", X_train.shape, "| Contexto:", X_contexto.shape, "| Validación:", X_valid.shape)


Training: (306255, 762) | Contexto: (306255, 762) | Validación: (131152, 762)


## 5. Modelo y validación

Se mantienen los pesos en FP32. El entrenamiento usa el AMP FP16 de la API; la evaluación completa usa FP32.
Se reutiliza el manejo de memoria probado: caché ICL en RAM y una capa por vez en GPU; temporales bajo `offload_raiz` del YAML, dentro de WSL.
Las consultas se reducen solo ante OOM, según el YAML; el contexto no se recorta. Cada evaluación reconstruye el caché porque cambian los pesos ICL.

In [10]:
trainer = FinetunedTabICLClassifier(
    device=str(DEVICE), random_state=SEED,
    n_estimators_finetune=PARAM["fijos"]["n_estimators_finetune"],
    n_estimators_inference=PARAM["fijos"]["n_estimators_inference"],
    freeze_col=PARAM["fijos"]["freeze_col"], freeze_row=PARAM["fijos"]["freeze_row"],
    **PARAM["fijos"]["preprocesamiento"],
)
trainer._load_pretrained(DEVICE)
modelo = trainer.model_.to(DEVICE)
for p in modelo.parameters():
    p.requires_grad_(True)
trainer._apply_freezing(modelo)
entrenables = [p for p in modelo.parameters() if p.requires_grad]
assert {id(p) for p in entrenables} == {id(p) for p in modelo.icl_predictor.parameters()}
pesos_iniciales = {n: t.detach().cpu().clone() for n, t in modelo.state_dict().items()}
print("Parámetros entrenables ICL:", sum(p.numel() for p in entrenables))


Parámetros entrenables ICL: 26276362


In [11]:
from tabicl._model.kv_cache import KVCache


class EntradaPreparada:
    def transform(self, X):
        return np.asarray(X)


class VistaPreparada:
    def __init__(self, dimension, metodo):
        self.metodo = metodo
        self.feature_shuffles_ = {metodo: [list(range(dimension))]}
        self.class_shuffles_ = {metodo: [[0, 1]]}

    def transform(self, X, mode="test"):
        return OrderedDict([(self.metodo, (np.asarray(X)[None],))])


class CacheICLRAM(KVCache):
    def to(self, device, dtype=None):
        # Evitar que el gestor suba todas las capas juntas a GPU.
        return CacheICLRAM(kv={
            i: entrada.to("cpu", dtype=dtype) for i, entrada in self.kv.items()
        })


def cargar_kv(modulo, entrada, argumentos):
    torch.cuda.synchronize(DEVICE)
    cache = argumentos.get("cached_kv")
    if cache is not None:
        # Copia temporal de una sola capa; el original permanece en RAM.
        return entrada, {**argumentos, "cached_kv": cache.to(DEVICE)}


def compactar_kv(modulo, entrada, salida):
    if isinstance(salida, tuple):
        # Conservar FP32 y copiar directamente a RAM, sin clonar en GPU.
        kv = [t.to(device="cpu", memory_format=torch.contiguous_format)
              for t in salida[1:]]
        torch.cuda.synchronize(DEVICE)
        capa = len(modelo._cache.icl_cache.kv) + 1
        total = len(modelo.icl_predictor.tf_icl.blocks)
        print(f"Caché ICL: {capa}/{total} capas en RAM.", flush=True)
        return (salida[0], *kv)
    torch.cuda.synchronize(DEVICE)


def preparar_cache(estimador, X, y, metodo):
    estimador._resolve_device()
    estimador._build_inference_config()
    estimador.n_samples_in_, estimador.n_features_in_ = X.shape
    estimador.y_encoder_ = LabelEncoder().fit(y)
    estimador.classes_ = estimador.y_encoder_.classes_
    estimador.n_classes_ = len(estimador.classes_)
    estimador.X_encoder_ = EntradaPreparada()
    estimador.ensemble_generator_ = VistaPreparada(X.shape[1], metodo)
    estimador.cache_mode_ = estimador.kv_cache
    entrada = torch.from_numpy(X[None]).float().to(DEVICE)
    etiquetas = torch.tensor(y[None], dtype=torch.float32, device=DEVICE)
    try:
        with ExitStack() as pila:
            def fin_filas(*args):
                torch.cuda.synchronize(DEVICE)
                print("Filas listas; construyendo caché ICL...", flush=True)
            pila.callback(modelo.row_interactor.register_forward_hook(fin_filas).remove)
            modelo.forward_with_cache(
                X_train=entrada, y_train=etiquetas,
                store_cache=True, cache_mode=estimador.cache_mode_,
                inference_config=estimador.inference_config_,
            )
    finally:
        del entrada, etiquetas
    modelo._cache.icl_cache = CacheICLRAM(kv=modelo._cache.icl_cache.kv)
    estimador.model_kv_cache_ = OrderedDict([(metodo, modelo._cache)])
    modelo.clear_cache()


In [12]:
from tqdm.auto import tqdm


@torch.no_grad()
def predecir(parametros):
    e = parametros["ejecucion"]
    metodo = parametros["fijos"]["preprocesamiento"]["norm_methods"]
    tamanos = e["tamanos_consulta_oom"]
    modo = modelo.training
    modelo.eval()
    modelo.zero_grad(set_to_none=True)
    modelo.clear_cache()
    gc.collect()
    torch.cuda.empty_cache()
    estimador = None
    raiz = Path(e["offload_raiz"]).expanduser()
    raiz.mkdir(parents=True, exist_ok=True)
    inicio = time.perf_counter()
    try:
        with tempfile.TemporaryDirectory(dir=raiz) as temporal, ExitStack() as pila:
            def fin_columnas(*args):
                print("Columnas listas; procesando filas...", flush=True)
            hook_filas = modelo.row_interactor.register_forward_pre_hook(fin_columnas)
            pila.callback(hook_filas.remove)
            for bloque in modelo.icl_predictor.tf_icl.blocks:
                pila.callback(bloque.register_forward_pre_hook(cargar_kv, with_kwargs=True).remove)
                pila.callback(bloque.register_forward_hook(compactar_kv).remove)
            comun = dict(use_amp=False, use_fa3=e["use_fa3"],
                         safety_factor=e["safety_factor"], use_async=e["use_async"])
            inferencia = {nombre: dict(comun) for nombre in ("COL_CONFIG", "ROW_CONFIG", "ICL_CONFIG")}
            inferencia["COL_CONFIG"].update(cpu_safety_factor=e["cpu_safety_factor"],
                max_pinned_memory_mb=e["max_pinned_memory_mb"], disk_offload_dir=temporal)
            estimador = trainer._build_inner_estimator(
                modelo, parametros["fijos"]["n_estimators_inference"], DEVICE
            )
            estimador.set_params(kv_cache=e["kv_cache"], batch_size=e["batch_size_api"],
                use_amp=False, use_fa3=e["use_fa3"], disk_offload_dir=temporal, inference_config=inferencia)
            with torch.autocast(device_type=DEVICE.type, enabled=False):
                print(f"Creando caché del contexto: {len(y_contexto):,} filas...", flush=True)
                preparar_cache(estimador, X_contexto, y_contexto, metodo)
                hook_filas.remove()
                torch.cuda.synchronize(DEVICE)
                print(f"Caché lista en {(time.perf_counter()-inicio)/60:.2f} min.", flush=True)
                probabilidades = np.empty(len(X_valid), dtype=np.float64)
                desde = indice_tamano = 0
                with tqdm(total=len(X_valid), desc="Validando", unit="filas") as progreso:
                    while desde < len(X_valid):
                        hasta = min(desde+tamanos[indice_tamano], len(X_valid))
                        try:
                            salida = estimador.predict_proba(X_valid[desde:hasta])
                            if not np.isfinite(salida).all():
                                raise FloatingPointError(f"Probabilidades no finitas en filas {desde}:{hasta}.")
                            probabilidades[desde:hasta] = salida[:, 1]
                        except torch.OutOfMemoryError as error:
                            traceback.clear_frames(error.__traceback__)
                            modelo.clear_cache()
                            gc.collect()
                            torch.cuda.empty_cache()
                            indice_tamano += 1
                            if indice_tamano == len(tamanos):
                                raise
                            tqdm.write(f"OOM de consulta: reintento con {tamanos[indice_tamano]} filas.")
                            continue
                        progreso.update(hasta-desde)
                        desde = hasta
                        del salida
            torch.cuda.synchronize(DEVICE)
            return probabilidades, time.perf_counter()-inicio
    except BaseException as error:
        # Liberar activaciones retenidas por el traceback; los pesos siguen en modelo.
        traceback.clear_frames(error.__traceback__)
        raise
    finally:
        if estimador is not None:
            estimador.model_kv_cache_ = None
            estimador.ensemble_generator_ = None
        modelo.clear_cache()
        modelo.train(modo)
        gc.collect()
        torch.cuda.empty_cache()


## 6. Entrenamiento

Épocas completas sobre la muestra fija. El tamaño del bloque, warmup, horizonte y LR salen del YAML.
Se conservan los defaults de AMP, clipping y partición contexto/consulta de la API.
El coseno termina en cero al completar el horizonte configurado, sin reinicios.

Se imprime avance al primer bloque, cada `log_cada_bloques` y al último, con loss media, LR y tiempo.
No hay monitoreo continuo de recursos.

In [13]:
def bloques_epoca(epoca, max_filas):
    rng = np.random.default_rng(np.random.SeedSequence([SEED, epoca]))
    partes = np.array_split(rng.permutation(len(y_train)), math.ceil(len(y_train)/max_filas))
    for i, posiciones in enumerate(partes):
        etiquetas = y_train[posiciones]
        if len(np.unique(etiquetas)) < 2 or np.bincount(etiquetas, minlength=2).min() < 2:
            raise ValueError("Bloque sin suficientes ejemplos de ambas clases.")
        batch = _build_meta_batch(
            X_train[posiciones], etiquetas, classification=True,
            n_estimators=trainer.n_estimators_finetune,
            query_size=max(2, int(len(posiciones)*trainer.finetune_ctx_query_ratio)),
            epoch_seed=SEED+epoca, chunk_idx=i, preprocessing_seed=SEED,
            norm_methods=trainer.norm_methods,
            feat_shuffle_method=trainer.feat_shuffle_method,
            class_shuffle_method=trainer.class_shuffle_method,
            outlier_threshold=trainer.outlier_threshold,
        )
        if trainer._task_skip_batch(batch):
            raise ValueError("Clase de consulta ausente en contexto; no se omiten filas.")
        yield batch, len(posiciones)



def entrenar(parametros, evaluar, avance):
    fijos = parametros["fijos"]
    modelo.load_state_dict(pesos_iniciales)
    fijar_semilla(SEED)
    optimizador = torch.optim.AdamW(
        entrenables, lr=parametros["hp"]["lr_maximo"], weight_decay=parametros["hp"]["weight_decay"]
    )
    _, scaler, amp_context = trainer._make_amp(DEVICE)
    pasos_epoca = math.ceil(len(y_train)/fijos["filas_por_bloque"])
    warmup = fijos["optimizacion"]["warmup_epochs"]*pasos_epoca
    total = fijos["epochs"]*pasos_epoca

    def factor_lr(paso):
        if paso < warmup:
            return paso/max(1, warmup)
        progreso = (paso-warmup)/max(1, total-warmup)
        return max(0.0, (1+math.cos(math.pi*progreso))/2)

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizador, factor_lr)
    for epoca in range(1, fijos["epochs"]+1):
        avance.update(fase="training", epoch=epoca)
        fijar_semilla(SEED+epoca)
        trainer._set_training_mode(True)
        modelo.clear_cache()
        torch.cuda.synchronize(DEVICE)
        inicio = time.perf_counter()
        vistas = updates = omitidos = 0
        suma_loss = 0.0
        for numero, (batch_cpu, n_filas) in enumerate(bloques_epoca(epoca, fijos["filas_por_bloque"]), 1):
            optimizador.zero_grad(set_to_none=True)
            batch = move_meta_batch(batch_cpu, DEVICE)
            with amp_context():
                loss = trainer._compute_batch_loss(batch, modelo)
            if not torch.isfinite(loss).item():
                raise FloatingPointError(f"Loss no finita en época {epoca}, bloque {numero}.")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizador)
            torch.nn.utils.clip_grad_norm_(entrenables, trainer.grad_clip)
            escala = scaler.get_scale()
            scaler.step(optimizador)
            scaler.update()
            scheduler.step()
            actualizo = scaler.get_scale() >= escala
            updates += int(actualizo)
            omitidos += int(not actualizo)
            vistas += n_filas
            suma_loss += float(loss.detach().item())
            del batch_cpu, batch, loss
            if numero == 1 or numero % parametros["ejecucion"]["log_cada_bloques"] == 0 or numero == pasos_epoca:
                segundos = time.perf_counter()-inicio
                print(f"Época {epoca}/{fijos['epochs']} | bloque {numero}/{pasos_epoca} | "
                      f"filas={vistas:,}/{len(y_train):,} | loss={suma_loss/numero:.4f} | "
                      f"LR={optimizador.param_groups[0]['lr']:.2e} | {segundos/60:.1f} min", flush=True)
        torch.cuda.synchronize(DEVICE)
        avance["segundos_train"] += time.perf_counter()-inicio
        assert vistas == len(y_train), "Época incompleta."
        if not updates:
            raise FloatingPointError(f"Ninguna actualización en época {epoca}.")
        print(f"Época {epoca}: {updates} actualizaciones, {omitidos} pasos omitidos AMP.", flush=True)
        if epoca in fijos["evaluar_epochs"]:
            avance["fase"] = "validacion"
            optimizador.zero_grad(set_to_none=True)
            evaluar(epoca, avance["segundos_train"])
            avance["fase"] = "training"


## 7. Búsqueda y log

`Grid_log.txt` queda en `directorios.exp / exp_name`. Al iniciar una búsqueda nueva se renombra el log anterior con fecha y hora.
Cada validación guarda AUC, HP, semilla, parámetros y tiempos: training acumulado de esa combinación, evaluación del corte y búsqueda acumulada.
La referencia de época 0 se calcula una vez y se repite en el log para comparar las combinaciones.

Un OOM o una loss no finita **durante training** se registra y permite continuar con la siguiente combinación.
Si falla una validación, se registra y se detiene la búsqueda: los pesos entrenados siguen en `modelo` para revisar o reintentar esa evaluación.
No se guardan pesos ni probabilidades. Los temporales de evaluación se eliminan al salir, también ante errores.

In [14]:
def loguear(iteracion, epoca, metrica, parametros, segundos_train, segundos_evaluacion,
             estado="ok", error=""):
    registro = {
        "fecha": datetime.now().isoformat(timespec="seconds"),
        "iter": iteracion, "epoch": epoca, "metrica": metrica,
        "semilla": SEED, **parametros["hp"],
        "segundos_train": segundos_train, "segundos_evaluacion": segundos_evaluacion,
        "segundos_busqueda": time.perf_counter()-inicio_busqueda,
        "estado": estado, "error": error,
        "parametros": json.dumps(parametros, ensure_ascii=False),
    }
    pd.DataFrame([registro]).to_csv(
        ARCHIVO_LOG, sep="\t", index=False, mode="a", header=not ARCHIVO_LOG.exists()
    )
    if estado == "ok":
        print(f"Validación | combinación={iteracion} | época={epoca} | AUC={metrica:.6f}", flush=True)
    else:
        print(f"{estado} | combinación={iteracion} | época={epoca} | {error}", flush=True)


In [15]:
if PARAM["fijos"]["correr_busqueda"]:
    if ARCHIVO_LOG.exists():
        ARCHIVO_LOG.rename(ARCHIVO_LOG.with_name(f"Grid_log_{datetime.now():%Y%m%d_%H%M%S_%f}.txt"))
    combinaciones = [dict(zip(hs, valores)) for valores in product(*hs.values())]
    print("Combinaciones:", len(combinaciones), flush=True)
    inicio_busqueda = time.perf_counter()
    metrica_0 = None
    segundos_0 = 0.0
    try:
        limitar_vram()
        if PARAM["fijos"]["evaluar_epoch_0"]:
            modelo.load_state_dict(pesos_iniciales)
            proba, segundos_0 = predecir(PARAM)
            metrica_0 = roc_auc_score(y_valid, proba)
            del proba
        for iteracion, hp in enumerate(combinaciones, 1):
            parametros = {**deepcopy(PARAM), "hp": hp}
            print(f"\nCombinación {iteracion}/{len(combinaciones)}: {hp}", flush=True)
            if metrica_0 is not None:
                loguear(iteracion, 0, metrica_0, parametros, 0.0, segundos_0 if iteracion == 1 else 0.0)

            def evaluar_validacion(epoca, segundos_train):
                proba, segundos = predecir(parametros)
                loguear(iteracion, epoca, roc_auc_score(y_valid, proba), parametros, segundos_train, segundos)

            avance = {"fase": "training", "epoch": 0, "segundos_train": 0.0}
            try:
                entrenar(parametros, evaluar_validacion, avance)
            except (torch.OutOfMemoryError, FloatingPointError) as error:
                loguear(iteracion, avance["epoch"], np.nan, parametros, avance["segundos_train"], 0.0,
                        estado=f"error_{avance['fase']}", error=f"{type(error).__name__}: {error}")
                if avance["fase"] == "validacion":
                    raise
                traceback.clear_frames(error.__traceback__)
                print("Se continúa con la siguiente combinación.", flush=True)
            finally:
                modelo.zero_grad(set_to_none=True)
                modelo.clear_cache()
                gc.collect()
                torch.cuda.empty_cache()
        print(f"Búsqueda finalizada: {(time.perf_counter()-inicio_busqueda)/3600:.2f} horas.", flush=True)
    finally:
        torch.cuda.set_per_process_memory_fraction(1.0, DEVICE)


Combinaciones: 6
Creando caché del contexto: 306,255 filas...
Columnas listas; procesando filas...
Filas listas; construyendo caché ICL...
Caché ICL: 1/12 capas en RAM.
Caché ICL: 2/12 capas en RAM.
Caché ICL: 3/12 capas en RAM.
Caché ICL: 4/12 capas en RAM.
Caché ICL: 5/12 capas en RAM.
Caché ICL: 6/12 capas en RAM.
Caché ICL: 7/12 capas en RAM.
Caché ICL: 8/12 capas en RAM.
Caché ICL: 9/12 capas en RAM.
Caché ICL: 10/12 capas en RAM.
Caché ICL: 11/12 capas en RAM.
Caché ICL: 12/12 capas en RAM.
Caché lista en 12.75 min.


Validando:   0%|          | 0/131152 [00:00<?, ?filas/s]


Combinación 1/6: {'lr_maximo': 1e-06, 'weight_decay': 0.0}
Validación | combinación=1 | época=0 | AUC=0.941654
Época 1/80 | bloque 1/75 | filas=4,084/306,255 | loss=0.1902 | LR=1.33e-08 | 0.0 min


/home/marco/anaconda3/envs/cmeyf-cuda13/lib/python3.11/site-packages/torch/autograd/graph.py:825: UserWarning: Flash Attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /opt/conda/conda-bld/pytorch_1729647348947/work/aten/src/ATen/native/transformers/cuda/attention_backward.cu:102.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Época 1/80 | bloque 20/75 | filas=81,680/306,255 | loss=0.1763 | LR=2.67e-07 | 0.3 min
Época 1/80 | bloque 40/75 | filas=163,350/306,255 | loss=0.1726 | LR=5.33e-07 | 0.7 min
Época 1/80 | bloque 60/75 | filas=245,010/306,255 | loss=0.1721 | LR=8.00e-07 | 1.0 min
Época 1/80 | bloque 75/75 | filas=306,255/306,255 | loss=0.1707 | LR=1.00e-06 | 1.3 min
Época 1: 75 actualizaciones, 0 pasos omitidos AMP.
Época 2/80 | bloque 1/75 | filas=4,084/306,255 | loss=0.1511 | LR=1.00e-06 | 0.0 min
Época 2/80 | bloque 20/75 | filas=81,680/306,255 | loss=0.1727 | LR=1.00e-06 | 0.3 min
Época 2/80 | bloque 40/75 | filas=163,350/306,255 | loss=0.1709 | LR=1.00e-06 | 0.7 min
Época 2/80 | bloque 60/75 | filas=245,010/306,255 | loss=0.1699 | LR=1.00e-06 | 1.0 min
Época 2/80 | bloque 75/75 | filas=306,255/306,255 | loss=0.1698 | LR=1.00e-06 | 1.2 min
Época 2: 75 actualizaciones, 0 pasos omitidos AMP.
Época 3/80 | bloque 1/75 | filas=4,084/306,255 | loss=0.1868 | LR=1.00e-06 | 0.0 min
Época 3/80 | bloque 20/75 

Validando:   0%|          | 0/131152 [00:00<?, ?filas/s]

... (43 outputs left out)

Validando:   0%|          | 0/131152 [00:00<?, ?filas/s]

Validación | combinación=6 | época=50 | AUC=0.918885
Época 51/80 | bloque 1/75 | filas=4,084/306,255 | loss=0.1371 | LR=3.15e-06 | 0.0 min
Época 51/80 | bloque 20/75 | filas=81,680/306,255 | loss=0.1479 | LR=3.11e-06 | 0.3 min
Época 51/80 | bloque 40/75 | filas=163,350/306,255 | loss=0.1514 | LR=3.06e-06 | 0.7 min
Época 51/80 | bloque 60/75 | filas=245,010/306,255 | loss=0.1536 | LR=3.01e-06 | 1.0 min
Época 51/80 | bloque 75/75 | filas=306,255/306,255 | loss=0.1536 | LR=2.97e-06 | 1.2 min
Época 51: 75 actualizaciones, 0 pasos omitidos AMP.
Época 52/80 | bloque 1/75 | filas=4,084/306,255 | loss=0.1490 | LR=2.97e-06 | 0.0 min
Época 52/80 | bloque 20/75 | filas=81,680/306,255 | loss=0.1509 | LR=2.92e-06 | 0.3 min
Época 52/80 | bloque 40/75 | filas=163,350/306,255 | loss=0.1519 | LR=2.88e-06 | 0.7 min
Época 52/80 | bloque 60/75 | filas=245,010/306,255 | loss=0.1518 | LR=2.83e-06 | 1.0 min
Época 52/80 | bloque 75/75 | filas=306,255/306,255 | loss=0.1531 | LR=2.79e-06 | 1.2 min
Época 52: 75 

Validando:   0%|          | 0/131152 [00:00<?, ?filas/s]

Validación | combinación=6 | época=80 | AUC=0.916780
Búsqueda finalizada: 19.53 horas.


## 8. Selección por AUC de validación

Se muestran todos los cortes válidos y se eligen HP/época por AUC. Ante empate se prefiere menor época y luego menor combinación, como en la referencia.
La época 0 participa como referencia del modelo sin fine-tuning; si resulta ganadora, la búsqueda no superó al original.
La selección no carga pesos de un corte anterior: conserva sus parámetros y época en `param_final` y `epoca_final` para la etapa posterior.

In [16]:
tb_grid = pd.read_csv(ARCHIVO_LOG, sep="\t")
tb_validos = tb_grid.loc[tb_grid["estado"].eq("ok") & tb_grid["metrica"].notna()].copy()
assert len(tb_validos), "No hay evaluaciones válidas en el log."
tb_validos = tb_validos.sort_values(["metrica", "epoch", "iter"], ascending=[False, True, True])
mejor = tb_validos.iloc[0]
param_final = json.loads(mejor["parametros"])
epoca_final = int(mejor["epoch"])
print("Mejores HP:", param_final["hp"])
print("Época:", epoca_final, "| AUC validación:", mejor["metrica"])
if epoca_final == 0:
    print("La búsqueda no superó al modelo original.")
display(tb_validos.pivot(index="iter", columns="epoch", values="metrica").sort_index())


Mejores HP: {'lr_maximo': 1e-06, 'weight_decay': 0.0}
Época: 0 | AUC validación: 0.9416536875989784
La búsqueda no superó al modelo original.


epoch,0,10,20,50,80
iter,,,,,
1,0.941654,0.936270,0.932358,0.928099,0.927070
2,0.941654,0.936267,0.932359,0.928103,0.927072
3,0.941654,0.930669,0.925104,0.921195,0.919187
4,0.941654,0.930670,0.925119,0.921225,0.919200
5,0.941654,0.924387,0.920351,0.918864,0.916758
6,0.941654,0.924377,0.920367,0.918885,0.916780
